<a href="https://colab.research.google.com/github/Zreyya/analisis-sentimen-spotify/blob/main/mesin_spotify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# STEP 1: LOAD DATASET
# =====================================================================
# Pastikan file 'ulasan_com.spotify.music.csv' sudah upload ke folder Colab
df = pd.read_csv('ulasan_com.spotify.music.csv')

# Bersihkan baris yang kosong pada kolom Ulasan atau Rating jika ada
df = df.dropna(subset=['Ulasan', 'Rating'])

print("--- Data Berhasil Dimuat ---")
print(f"Total data awal: {len(df)} baris")
print(df[['Ulasan', 'Rating']].head())

--- Data Berhasil Dimuat ---
Total data awal: 100000 baris
                                              Ulasan  Rating
0                            suka banget sama ni app       5
1                                               bgus       5
2  ga bisa denger lagu JKT48 lagi gara² selalu di...       1
3            lagunya bagus banget buat ngapa ngapain       5
4  kenapa sih, aku cuman bisa main 5 lagu doang? ...       1


In [ ]:
# STEP 2: PELABELAN OTOMATIS (SENTIMEN)
# =====================================================================
# dataset belum punya kolom sentimen (positif/negatif), buat berdasarkan Rating.
# Rating 4 dan 5 = Positif (1)
# Rating 1 dan 2 = Negatif (0)
# Rating 3 (Netral) abaikan dulu supaya klasifikasinya lebih tegas.

df = df[df['Rating'] != 3]
df['Sentimen'] = df['Rating'].apply(lambda x: 1 if x > 3 else 0)

print("\n--- Proses Pelabelan Selesai ---")
print(df['Sentimen'].value_counts())


--- Proses Pelabelan Selesai ---
Sentimen
1    75670
0    19689
Name: count, dtype: int64


In [ ]:
# STEP 3: TEXT PREPROCESSING (PEMBERSIHAN TEKS)
# =====================================================================
# Mengubah teks menjadi huruf kecil semua dan menghapus tanda baca/emoji
def bersihkan_teks(teks):
    teks = str(teks).lower() # Huruf kecil semua
    teks = re.sub(r'[^a-zA-Z0-9\s]', '', teks) # Hapus simbol, emoji, dan tanda baca
    return teks

df['Ulasan_Clean'] = df['Ulasan'].apply(bersihkan_teks)

print("\n--- Hasil Pembersihan Teks (3 Baris Pertama) ---")
print(df[['Ulasan', 'Ulasan_Clean']].head(3))


--- Hasil Pembersihan Teks (3 Baris Pertama) ---
                                              Ulasan  \
0                            suka banget sama ni app   
1                                               bgus   
2  ga bisa denger lagu JKT48 lagi gara² selalu di...   

                                        Ulasan_Clean  
0                            suka banget sama ni app  
1                                               bgus  
2  ga bisa denger lagu jkt48 lagi gara selalu di ...  


In [ ]:
# STEP 4: MEMBAGI DATA (TRAIN & TEST SPLIT)
# =====================================================================
# X adalah fiturnya (teks ulasan), y adalah targetnya (sentimen)
X = df['Ulasan_Clean']
y = df['Sentimen']

In [ ]:
# Kita bagi 80% untuk training dan 20% untuk testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n--- Pembagian Data Selesai ---")
print(f"Data Latih (Train): {len(X_train)} baris")
print(f"Data Uji (Test): {len(X_test)} baris")


--- Pembagian Data Selesai ---
Data Latih (Train): 76287 baris
Data Uji (Test): 19072 baris


In [ ]:
# STEP 5: VEKTORISASI TEKS (MENGUBAH TEKS MENJADI ANGKA)
# =====================================================================
# Komputer tidak paham teks langsung, jadi kita ubah dengan TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# Fit dan transform pada data training, lalu transform pada data testing
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
# STEP 6: TRAINING MODEL NAIVE BAYES
# =====================================================================
# Menggunakan MultinomialNB karena ini adalah kasus klasifikasi teks
model_nb = MultinomialNB()
model_nb.fit(X_train_vec, y_train)

print("\n--- Proses Training Selesai! ---")


--- Proses Training Selesai! ---


In [ ]:
# STEP 7: EVALUASI MODEL
# =====================================================================
# Prediksi data uji menggunakan model yang sudah dilatih
y_pred = model_nb.predict(X_test_vec)

In [ ]:
# Menampilkan laporan klasifikasi detail (Precision, Recall, F1-Score)
print("\nLaporan Klasifikasi Detail:")
print(classification_report(y_test, y_pred, target_names=['Negatif', 'Positif']))


Laporan Klasifikasi Detail:
              precision    recall  f1-score   support

     Negatif       0.86      0.68      0.76      4033
     Positif       0.92      0.97      0.94     15039

    accuracy                           0.91     19072
   macro avg       0.89      0.83      0.85     19072
weighted avg       0.91      0.91      0.91     19072



In [ ]:
# Menghitung nilai akurasi
akurasi = accuracy_score(y_test, y_pred)
print(f"AKURASI MODEL: {akurasi * 100:.2f}%")

AKURASI MODEL: 91.02%
